# OpsPilot Ticket Intelligence: Notebook Walkthrough

This notebook puts the current ML pipeline in one place so you can run it, inspect it, and improve it little by little.

What it covers:

- loading the ticket dataset
- auditing labels and text
- explaining the current cleaning/normalization
- training the TF-IDF + Logistic Regression baseline
- evaluating category and priority models
- viewing confusion matrices and error examples
- running escalation-risk and routing logic
- producing the final model-versioned JSON output
- trying normalization and hyperparameter experiments
- understanding the transformer-ready next step

Current design philosophy: the model predicts category/priority, while rules and confidence thresholds handle escalation routing and human review.

## 1. Setup

Run this notebook from either:

- `verticals/ticket-intelligence/notebooks/`, or
- `verticals/ticket-intelligence/`

The path helper below finds the vertical root and dataset automatically.

In [ ]:
from __future__ import annotations

import json
import pickle
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline


def find_vertical_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        data_path = candidate / "ml" / "ticket_intelligence" / "data" / "synthetic_tickets.csv"
        if data_path.exists():
            return candidate
    raise FileNotFoundError("Could not find vertical root containing ml/ticket_intelligence/data/synthetic_tickets.csv")


VERTICAL_ROOT = find_vertical_root()
DATA_PATH = VERTICAL_ROOT / "ml" / "ticket_intelligence" / "data" / "synthetic_tickets.csv"
OUTPUT_DIR = VERTICAL_ROOT / "ml" / "ticket_intelligence" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_VERSION = "ticket-intelligence-v1"
RANDOM_STATE = 42

print("Vertical root:", VERTICAL_ROOT)
print("Data path:", DATA_PATH)

## 2. Load The Data

The current dataset is synthetic. It is intentionally small because the goal is to prove the product-style pipeline end to end:

`ticket_id`, `customer_message`, `category`, and `priority`.

This is not production data. Treat it as a starter training/evaluation sandbox.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head(10)

In [ ]:
print("Category distribution")
display(df["category"].value_counts().rename_axis("category").reset_index(name="rows"))

print("Priority distribution")
display(df["priority"].value_counts().rename_axis("priority").reset_index(name="rows"))

print("Missing values")
display(df.isna().sum().rename("missing_count"))

## 3. Cleaning And Normalization

The production baseline currently uses light normalization inside `TfidfVectorizer`:

- `lowercase=True`
- `strip_accents="unicode"`
- `ngram_range=(1, 2)`

It does **not** currently remove stop words, stem words, lemmatize words, or aggressively remove punctuation.

Why keep it light? In ticket triage, words like `not`, `again`, `today`, `refund`, `legal`, `blocked`, and `cancel` are important. Too much cleaning can erase operational signal.

The optional function below is for experiments. You can turn this on and compare metrics against the current baseline.

In [ ]:
def normalize_text(text: str) -> str:
    """Optional experiment normalizer.

    Current saved baseline does not manually apply this. It is here so you can
    test whether stronger normalization improves or hurts the model.
    """
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " <URL> ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.\w+\b", " <EMAIL> ", text)
    text = re.sub(r"\b\d+\b", " <NUMBER> ", text)
    text = re.sub(r"([!?.,;:/()\[\]{}])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


sample = df.loc[0, "customer_message"]
print("Original:", sample)
print("Normalized:", normalize_text(sample))

## 4. Train/Test Split

We use a 75/25 train/test split and stratify by `category` so each category is represented in the test set.

Note: priority is not used for stratification today. With more data, a better split could stratify jointly by category and priority or use cross-validation.

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=df["category"],
)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

display(train_df[["ticket_id", "customer_message", "category", "priority"]].head())

## 5. Baseline Model: TF-IDF + Logistic Regression

We train two separate models:

1. Category model: `customer_message -> category`
2. Priority model: `customer_message -> priority`

The model is intentionally simple and explainable. It gives us a benchmark before trying transformers.

In [ ]:
def build_baseline_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    min_df=1,
                    max_features=5000,
                    strip_accents="unicode",
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


category_model = build_baseline_pipeline()
priority_model = build_baseline_pipeline()

category_model.fit(train_df["customer_message"], train_df["category"])
priority_model.fit(train_df["customer_message"], train_df["priority"])

print("Models trained")

## 6. Evaluation Helpers

We care about more than accuracy.

- Accuracy: overall correctness
- Macro-F1: treats every class equally, useful when classes are imbalanced
- Weighted-F1: weighted by support
- Per-class precision/recall/F1: shows which ticket types are weak
- Confusion matrix: shows where mistakes go
- Error examples: lets you inspect real misses

In [ ]:
def evaluate_model(name: str, model: Pipeline, x_test: pd.Series, y_test: pd.Series) -> tuple[dict, list[str]]:
    predictions = model.predict(x_test)
    labels = sorted(y_test.unique().tolist())
    metrics = {
        "target": name,
        "accuracy": round(float(accuracy_score(y_test, predictions)), 4),
        "macro_f1": round(float(f1_score(y_test, predictions, average="macro")), 4),
        "weighted_f1": round(float(f1_score(y_test, predictions, average="weighted")), 4),
        "labels": labels,
        "classification_report": classification_report(
            y_test, predictions, labels=labels, output_dict=True, zero_division=0
        ),
        "confusion_matrix": confusion_matrix(y_test, predictions, labels=labels).tolist(),
    }
    return metrics, predictions.tolist()


category_metrics, category_predictions = evaluate_model(
    "category", category_model, test_df["customer_message"], test_df["category"]
)
priority_metrics, priority_predictions = evaluate_model(
    "priority", priority_model, test_df["customer_message"], test_df["priority"]
)

summary = {
    "model_version": MODEL_VERSION,
    "baseline": "tfidf_logistic_regression",
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "category_accuracy": category_metrics["accuracy"],
    "category_macro_f1": category_metrics["macro_f1"],
    "priority_accuracy": priority_metrics["accuracy"],
    "priority_macro_f1": priority_metrics["macro_f1"],
}
summary

In [ ]:
print("Category classification report")
display(pd.DataFrame(category_metrics["classification_report"]).T)

print("Priority classification report")
display(pd.DataFrame(priority_metrics["classification_report"]).T)

In [ ]:
def plot_confusion(metrics: dict, title: str) -> None:
    labels = metrics["labels"]
    matrix = metrics["confusion_matrix"]
    fig, ax = plt.subplots(figsize=(8, 6))
    image = ax.imshow(matrix, interpolation="nearest", cmap="Blues")
    fig.colorbar(image, ax=ax)
    ax.set(
        xticks=range(len(labels)),
        yticks=range(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        ylabel="True label",
        xlabel="Predicted label",
        title=title,
    )
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right", rotation_mode="anchor")
    for row_idx, row in enumerate(matrix):
        for col_idx, value in enumerate(row):
            ax.text(col_idx, row_idx, value, ha="center", va="center", color="black")
    fig.tight_layout()
    plt.show()


plot_confusion(category_metrics, "Category Confusion Matrix")
plot_confusion(priority_metrics, "Priority Confusion Matrix")

## 7. Error Analysis

This table shows test examples where either category or priority was wrong.

This is where most practical ML improvement starts. Read the misses and ask:

- Was the label ambiguous?
- Did the text need better normalization?
- Is the model missing domain terms?
- Are categories overlapping?
- Is priority labeling inconsistent?

In [ ]:
error_df = test_df[["ticket_id", "customer_message", "category", "priority"]].copy()
error_df["predicted_category"] = category_predictions
error_df["predicted_priority"] = priority_predictions
error_df["category_correct"] = error_df["category"] == error_df["predicted_category"]
error_df["priority_correct"] = error_df["priority"] == error_df["predicted_priority"]
error_examples = error_df[(~error_df["category_correct"]) | (~error_df["priority_correct"])]

display(error_examples)

## 8. Save Artifacts

The modular scripts already save these files. This notebook can do the same so your experiments are easy to compare.

In [ ]:
metrics = {
    "model_version": MODEL_VERSION,
    "baseline": "tfidf_logistic_regression_notebook",
    "dataset": str(DATA_PATH.relative_to(VERTICAL_ROOT)),
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "category": category_metrics,
    "priority": priority_metrics,
}

with (OUTPUT_DIR / "notebook_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)
with (OUTPUT_DIR / "notebook_category_model.pkl").open("wb") as handle:
    pickle.dump(category_model, handle)
with (OUTPUT_DIR / "notebook_priority_model.pkl").open("wb") as handle:
    pickle.dump(priority_model, handle)
error_examples.to_csv(OUTPUT_DIR / "notebook_error_analysis.csv", index=False)

print("Saved notebook artifacts to", OUTPUT_DIR)

## 9. Escalation Risk And Routing

Escalation risk is rule-based today. This is deliberate: it is transparent, easy to audit, and safe for a first human-in-the-loop version.

The risk model looks for signals like angry language, refund/billing disputes, urgency, repeat issue language, and legal/compliance words.

In [ ]:
ANGRY_TERMS = {
    "angry", "furious", "unacceptable", "ignored", "terrible", "escalating", "escalate", "vp", "executive",
}
BILLING_TERMS = {"refund", "charged", "charge", "billing", "invoice", "dispute", "credit", "reimbursement"}
URGENCY_TERMS = {"today", "immediately", "urgent", "blocked", "blocking", "production", "deadline", "one hour"}
REPEAT_TERMS = {"again", "second", "twice", "three calls", "repeat", "keeps", "already"}
LEGAL_TERMS = {"legal", "attorney", "compliance", "gdpr", "hipaa", "breach", "contract", "dpa", "soc 2"}
SENSITIVE_CATEGORIES = {"compliance_request", "billing_dispute", "cancellation"}


@dataclass(frozen=True)
class RoutingResult:
    escalation_risk: float
    routing_decision: str
    reason: str
    risk_signals: list[str]


def contains_any(text: str, terms: Iterable[str]) -> list[str]:
    found = []
    for term in terms:
        pattern = r"\b" + re.escape(term) + r"\b"
        if re.search(pattern, text):
            found.append(term)
    return sorted(found)


def score_escalation_risk(text: str, category: str | None = None, priority: str | None = None) -> tuple[float, list[str]]:
    normalized = text.lower()
    signals = []
    score = 0.12
    signal_groups = [
        ("angry language", ANGRY_TERMS, 0.19),
        ("billing/refund dispute", BILLING_TERMS, 0.15),
        ("urgency/business impact", URGENCY_TERMS, 0.18),
        ("repeat issue", REPEAT_TERMS, 0.14),
        ("legal/compliance sensitivity", LEGAL_TERMS, 0.20),
    ]
    for label, terms, weight in signal_groups:
        matches = contains_any(normalized, terms)
        if matches:
            score += weight
            signals.append(f"{label}: {', '.join(matches[:3])}")
    if category in SENSITIVE_CATEGORIES:
        score += 0.08
        signals.append(f"sensitive category: {category}")
    if priority == "high":
        score += 0.12
        signals.append("high predicted priority")
    elif priority == "medium":
        score += 0.05
        signals.append("medium predicted priority")
    return round(min(score, 0.99), 2), signals


def route_ticket(
    text: str,
    category: str,
    priority: str,
    confidence: float,
    low_confidence_threshold: float = 0.58,
    high_risk_threshold: float = 0.72,
) -> RoutingResult:
    risk, signals = score_escalation_risk(text, category, priority)
    if category == "compliance_request" and risk >= 0.55:
        decision = "supervisor_review"
        reason = "Policy-sensitive ticket with compliance/legal signals"
    elif confidence < low_confidence_threshold:
        decision = "human_review"
        reason = "Low model confidence requires human validation"
    elif risk >= high_risk_threshold:
        decision = "priority_queue"
        reason = "High escalation risk based on language, impact, or sensitive terms"
    elif priority == "high":
        decision = "human_review"
        reason = "High-priority ticket should be checked before action"
    else:
        decision = "auto_triage_suggestion"
        reason = "High enough confidence with low escalation risk"
    return RoutingResult(risk, decision, reason, signals)

## 10. Final Prediction Function

This creates the same shape of output the API returns.

It also includes the low-confidence lexical fallback. If model confidence is very low but category-specific terms are strong, we can override the category while preserving the raw model category in metadata.

In [ ]:
CATEGORY_HINTS = {
    "billing_dispute": {"refund", "charged", "charge", "invoice", "billing", "dispute", "credit", "reimbursement"},
    "technical_issue": {"api", "error", "failed", "crash", "500", "webhook", "integration", "dashboard", "sync"},
    "account_access": {"login", "locked", "password", "mfa", "sso", "admin", "invite", "user", "portal"},
    "cancellation": {"cancel", "cancelled", "cancellation", "downgrade", "renewal", "subscription", "leaving"},
    "shipping_delay": {"shipment", "package", "tracking", "delivery", "delivered", "carrier", "parcel", "replacement"},
    "compliance_request": {"legal", "compliance", "gdpr", "hipaa", "soc 2", "dpa", "privacy", "audit", "breach"},
}


def predict_with_confidence(model: Pipeline, text: str) -> tuple[str, float, dict[str, float]]:
    label = str(model.predict([text])[0])
    probabilities = model.predict_proba([text])[0]
    classes = [str(value) for value in model.classes_]
    distribution = {klass: round(float(prob), 4) for klass, prob in zip(classes, probabilities)}
    return label, round(float(max(probabilities)), 4), distribution


def category_hint(text: str) -> tuple[str | None, int]:
    normalized = text.lower()
    scores = {category: sum(1 for term in terms if term in normalized) for category, terms in CATEGORY_HINTS.items()}
    category, score = max(scores.items(), key=lambda item: item[1])
    return (category, score) if score >= 2 else (None, score)


def predict_ticket(text: str, ticket_id: str | None = None) -> dict:
    raw_category, category_confidence, category_distribution = predict_with_confidence(category_model, text)
    priority, priority_confidence, priority_distribution = predict_with_confidence(priority_model, text)
    hinted_category, hint_score = category_hint(text)
    category_source = "model"
    category = raw_category
    if category_confidence < 0.4 and hinted_category and hinted_category != raw_category:
        category = hinted_category
        category_source = "low_confidence_lexical_fallback"
    confidence = round(min(category_confidence, priority_confidence), 4)
    routing = route_ticket(text, category, priority, confidence)
    return {
        "ticket_id": ticket_id,
        "category": category,
        "priority": priority,
        "escalation_risk": routing.escalation_risk,
        "confidence": confidence,
        "routing_decision": routing.routing_decision,
        "reason": routing.reason,
        "risk_signals": routing.risk_signals,
        "model_version": MODEL_VERSION,
        "model_family": "tfidf_logistic_regression_baseline",
        "metadata": {
            "raw_model_category": raw_category,
            "category_source": category_source,
            "category_hint_score": hint_score,
            "category_confidence": category_confidence,
            "priority_confidence": priority_confidence,
            "category_distribution": category_distribution,
            "priority_distribution": priority_distribution,
        },
    }


demo_text = "I was charged twice and support keeps closing my tickets. If this refund is not handled today I am escalating to legal."
result = predict_ticket(demo_text, ticket_id="TCK-NOTEBOOK-001")
print(json.dumps(result, indent=2))

## 11. Optional Experiment: Stronger Normalization

This trains the same baseline on a manually normalized text column. Compare the metrics with the original baseline.

If scores improve, normalization may be worth moving into the production training script. If scores fall, the current light cleaning is probably better for this dataset.

In [ ]:
normalized_df = df.copy()
normalized_df["normalized_message"] = normalized_df["customer_message"].map(normalize_text)

norm_train_df, norm_test_df = train_test_split(
    normalized_df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=normalized_df["category"],
)

norm_category_model = build_baseline_pipeline()
norm_priority_model = build_baseline_pipeline()
norm_category_model.fit(norm_train_df["normalized_message"], norm_train_df["category"])
norm_priority_model.fit(norm_train_df["normalized_message"], norm_train_df["priority"])

norm_category_metrics, _ = evaluate_model(
    "normalized_category", norm_category_model, norm_test_df["normalized_message"], norm_test_df["category"]
)
norm_priority_metrics, _ = evaluate_model(
    "normalized_priority", norm_priority_model, norm_test_df["normalized_message"], norm_test_df["priority"]
)

pd.DataFrame(
    [
        {"experiment": "baseline", "target": "category", "accuracy": category_metrics["accuracy"], "macro_f1": category_metrics["macro_f1"]},
        {"experiment": "normalized_text", "target": "category", "accuracy": norm_category_metrics["accuracy"], "macro_f1": norm_category_metrics["macro_f1"]},
        {"experiment": "baseline", "target": "priority", "accuracy": priority_metrics["accuracy"], "macro_f1": priority_metrics["macro_f1"]},
        {"experiment": "normalized_text", "target": "priority", "accuracy": norm_priority_metrics["accuracy"], "macro_f1": norm_priority_metrics["macro_f1"]},
    ]
)

## 12. Optional Experiment: Hyperparameter Tuning

This is a small grid search. On this tiny dataset, results may be noisy, but it shows the correct workflow.

Try expanding this once you have more labeled tickets.

In [ ]:
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "tfidf__max_features": [1000, 3000, 5000],
    "clf__C": [0.3, 1.0, 3.0, 10.0],
    "clf__class_weight": ["balanced", None],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
category_search = GridSearchCV(
    estimator=build_baseline_pipeline(),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
)
category_search.fit(train_df["customer_message"], train_df["category"])

print("Best category CV macro-F1:", round(category_search.best_score_, 4))
print("Best params:", category_search.best_params_)

best_category_model = category_search.best_estimator_
best_category_metrics, _ = evaluate_model("category_tuned", best_category_model, test_df["customer_message"], test_df["category"])
print("Held-out tuned category metrics:", {k: best_category_metrics[k] for k in ["accuracy", "macro_f1", "weighted_f1"]})

## 13. Transformer-Ready Next Step

The repo includes `train_transformer.py`, which is ready for DistilBERT/MiniLM-style fine-tuning when `torch` and `transformers` are installed.

For today, the right sequence is:

1. Get the baseline working.
2. Improve labels and data volume.
3. Tune normalization/hyperparameters.
4. Calibrate confidence.
5. Fine-tune a lightweight transformer and compare macro-F1.

Do not train one huge model to do classification, risk, routing, policy retrieval, and response generation all at once. Keep the system modular.

In [ ]:
def dependency_status(packages=("torch", "transformers", "datasets", "evaluate")) -> dict[str, str]:
    status = {}
    for package in packages:
        try:
            module = __import__(package)
            status[package] = getattr(module, "__version__", "installed")
        except Exception as exc:
            status[package] = f"missing: {exc.__class__.__name__}"
    return status


dependency_status()

## 14. Improvement Roadmap

Best next moves:

1. **Data**: add real historical tickets and final resolved labels.
2. **Priority labels**: define `low`, `medium`, `high` with strict rules.
3. **Normalization experiments**: compare raw text, light normalization, and structured replacements.
4. **Hyperparameters**: tune n-grams, max features, regularization, and class weights.
5. **Calibration**: use `CalibratedClassifierCV` and tune human-review thresholds.
6. **Risk labels**: train escalation risk once you have actual escalation outcomes.
7. **Transformer**: fine-tune DistilBERT/MiniLM after you have enough examples.
8. **Human feedback**: store overrides so the model improves from review decisions.
9. **Monitoring**: track macro-F1, confidence, escalation false negatives, and override rate over time.